# Recourse under predictive multiplicity — V7

A self-contained package with all six datasets in `dataset/`, implementation in `utils/`, and one notebook. No prior experimental results are required. All sections use the single **quick** mode control below. V7 removes the separate objective-loss convergence experiment and keeps duality-gap experiments for logistic, mixed linear/neural, and decision-tree competing sets.

The main deterministic gap is evaluated at the arithmetic average of oracle responses and average pre-update adversary weights. The uniform-random-response gap is saved separately. Convexity justifies averaging features only in the affine-logit experiment; neural and tree average-point gaps are empirical diagnostics.

Run from the extracted package with Python 3.11 and dependencies from `requirements.txt`. Tree experiments need Gurobi; mixed neural gap bounds need a suitable full-size license. To run only differentiable comparisons and convex gaps without Gurobi, turn both `RUN_TREES` and `RUN_MIXED_GAP` off. No model is replaced by a surrogate merely to bypass a missing solver.


## 1. Locate the package
Install dependencies once using `python -m pip install -r requirements.txt`. If needed, uncomment the installation line, run it once, then restart the kernel. Start Jupyter in this package directory or set `ROOT` explicitly.


In [16]:
# %pip install -r requirements.txt
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "RecourseModelMultiplicity-v7.ipynb").is_file():
    candidates = [ROOT / "RecourseModelMultiplicity-v7", *ROOT.parents]
    ROOT = next((p for p in candidates if (p / "RecourseModelMultiplicity-v7.ipynb").is_file()), ROOT)
if not (ROOT / "utils" / "recourse_runtime.py").is_file() or not (ROOT / "dataset" / "manifest.json").is_file():
    raise FileNotFoundError("Set ROOT to the V7 folder containing dataset/, utils/, and this notebook.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "results" / ".matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Package:", ROOT)

# Optional location for later historical comparisons; V7 does not write here or load it automatically.
LEGACY_RESULTS_ROOT = Path(os.environ.get("RECOURSE_LEGACY_RESULTS",
    "/Users/sj987/Documents/ChatGPT/Code/Predictive_Multiplicity_Recourse/results_corrected"))
print("V4/V5 results:", LEGACY_RESULTS_ROOT, "(available)" if LEGACY_RESULTS_ROOT.is_dir() else "(optional; not found)")


Package: /Users/sj987/Documents/ChatGPT/Code/Predictive_Multiplicity_Recourse/V7
V4/V5 results: /Users/sj987/Documents/ChatGPT/Code/Predictive_Multiplicity_Recourse/results_corrected (optional; not found)


## 2. One control cell for all experiments

The defaults are **quick mode, diversity enabled, and majority rejection**. Set `CANDIDATE_SELECTION` to one of:
- **`"reference"`**: the designated reference model rejects the held-out individual. By default this is the first selected model, pinned when feature diversity is enabled. Other models may accept the individual.
- **`"majority"`**: at least half of the actual competing models reject the individual. Ties count: 3 of 6 qualifies; 2 of 5 does not. The reference model need not reject the individual. A random forest counts as one competing model, irrespective of its internal number of trees.

A model rejects an individual when its favorable-class probability is **strictly below 0.5**. A probability equal to 0.5 is accepted. Selection uses held-out predictions before recourse optimization, not the test labels or recourse success. If too few individuals qualify, the run records the smaller count; it never fills the cohort with ineligible people.

`diversity=True` trains validation-accuracy-matched feature-subset replacements, including actionable feature groups. The reference model remains pinned. This applies to differentiable experiments and decision-tree gaps. Existing tree/forest baseline experiments retain their V6 training procedure. Model masks, admission records, eligible counts, selected IDs, and rejection counts are saved.

**Quick defaults:** Synthetic and COMPAS; 2 seeds; up to 3 individuals per seed; 20 rounds for logistic/mixed gaps; 10 for tree gaps; 10 ORPM comparison rounds; 5 tree comparison rounds; checked classifier training; 25 inner gradient steps for comparison oracles. Mixed trade-offs retain (2,2,2), (4,4,4), and (8,8,8), with three budgets per dataset. `"intermediate"` and `"full"` remain available but are not enabled.

Results are separated by version, run mode, selection rule, feature-diversity condition, experiment, and unique run ID. `RESUME=True` reuses only compatible completed jobs. Changing the selection rule or mode cannot reuse incompatible outcomes.


In [17]:
RUN_MODE = "quick"
CANDIDATE_SELECTION = "majority"  # "majority" (ties included) or "reference"
SEED = 42
diversity = True
RESUME = True
RUN_TREES = True
RUN_MIXED_GAP = True
RUN_PAIRED_DIVERSITY_CHECK = False  # Optional extra off/on runs; uses RUN_MODE too.
INCLUDE_NEURAL_MIQCP = None  # Baseline reference: off in quick mode.
SHOW_PLOTS = True

from IPython.display import display
# Restart the kernel after replacing utils so no older modules remain loaded.
from utils import recourse_training as training
assert training.PROTOCOL == "checked_classifier_training_v2", "Restart the kernel after updating V7."
from utils import recourse_core as rc
assert getattr(rc, "TRAINING_PROTOCOL", None) == training.PROTOCOL, "Restart the kernel: older core utilities are still loaded."
from utils import recourse_runtime as runtime
from utils import recourse_campaign as campaign
assert campaign.PROFILE_REVISION == "V7-selection-duality-training-v2", "Restart the kernel: older campaign utilities are still loaded."

inventory, workload = runtime.initialize(
    mode=RUN_MODE, seed=SEED, run_trees=RUN_TREES,
    run_mixed_gap=RUN_MIXED_GAP, candidate_selection=CANDIDATE_SELECTION,
    include_neural_miqcp=INCLUDE_NEURAL_MIQCP, root=ROOT, diversity=diversity,
)
display(inventory)
display(workload)


AssertionError: Restart the kernel: older core utilities are still loaded.

### Classifier training checks

Restart the kernel once after installing these utilities, then run this notebook from the top.
All three modes use the same classifier training criteria. Quick mode reduces the experimental workload; it does not accept a classifier merely because a short epoch budget ended.

Logistic regression minimizes mean binary cross-entropy plus L2 regularization (0.001, including the intercept), with a training-objective error bound at most 1e-8. Sampled logistic hypotheses instead pass a regularized loss-excess bound and validation-accuracy filtering. Neural networks use up to 100 Adam warmup epochs, then L-BFGS refinement and continuation checks; admission requires a measured gradient norm at most 0.001 or two sufficiently small continuation improvements. These neural checks are empirical, not a global convergence guarantee. Failed candidates are saved and excluded; an insufficient competing set is reported explicitly.

These are classifier training checks, separate from recourse-oracle optimization and ORPM duality-gap convergence. Tree/forest fitting uses its existing finite construction procedure. Every run saves the training protocol and per-model checks. Resume requires matching source, settings, and data, so runs made with the earlier fixed-epoch utilities are not reused. Earlier results remain in `results/`.


## 3. Convex logistic duality gap
Use affine-logit logistic models and normalized cross-entropy. Save lower and upper gap bounds, raw cross-entropy gaps, average recourses, all response/weight trajectories, and pointwise confidence intervals over seed means. Quick uses three logistic models.


In [ ]:
duality_gap = runtime.run_section("duality_gap", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
display(runtime.compact_summary(duality_gap))


## 4. Mixed linear/neural duality gap
Use one logistic, one single-hidden-layer, and one two-hidden-layer model in quick mode; larger modes use two of each. Cross-entropy composed with the neural networks is nonconvex in the recourse vector. A gradient-based local oracle supplies feasible responses; a separate mixed-integer ReLU encoding with logistic-loss lower approximations bounds the dual objective. A local minimum is never treated as a global lower bound. Wide optimization intervals remain visible.


In [ ]:
if RUN_MIXED_GAP:
    mixed_gap = runtime.run_section("mixed_duality_gap", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
    display(runtime.compact_summary(mixed_gap))
else:
    mixed_gap = None
    print("Disabled by RUN_MIXED_GAP=False.")


## 5. Homogeneous logistic baselines
Compare ORPM, ADV, ElliCE, and ROAR on the same model set, eligible individuals, budgets, and feature constraints.


In [ ]:
linear_results = runtime.run_section("linear_baselines", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
display(runtime.compact_summary(linear_results))


## 6. Unseen-model generalization
Generate recourse against an optimization set and evaluate it on separately generated unseen models. Reference/majority selection uses only the optimization set. Unseen models do not determine eligibility.


In [ ]:
generalization_results = runtime.run_section("generalization", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
display(runtime.compact_summary(generalization_results))


## 7. Homogeneous neural-network baselines
Compare methods for independently trained neural networks with (50,100) hidden-layer widths. Quick uses three networks. The optional neural MIQCP baseline is off in quick mode.


In [ ]:
neural_results = runtime.run_section("nonlinear_baselines", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
display(runtime.compact_summary(neural_results))


## 8. Budget, joint validity, and competing-set size
Use (2,2,2), (4,4,4), and (8,8,8) linear/single-layer/double-layer model counts. Save budget-versus-validity and model-count-versus-validity plots for each dataset. Model sets remain nested, and each cohort is reused across budgets and methods. Reference-mode cohorts remain fixed across nested sizes; majority-mode cohorts can change with set size. Consequently, majority-mode size curves combine changes in model sets and population eligibility. Saved cohort IDs and counts make this distinction auditable. Joint validity means every competing model accepts the generated recourse.


In [ ]:
tradeoff_results = runtime.run_section("cost_validity", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
display(runtime.compact_summary(tradeoff_results))


## 9. Burden of multiplicity — differentiable models
Use the trade-offs from the preceding cell. Search for recourse for each competing model individually and compare achieved singleton validity with saved joint validity. Source factuals, model sets, seeds, and the selected cohort rule are preserved. The worst individual model is selected within each seed; uncertainty is computed across paired seed differences. Search failure is not proof of infeasibility.


In [ ]:
if "tradeoff_results" not in globals():
    raise RuntimeError("Run the cost-validity cell first.")
multiplicity_results = runtime.run_burden(tradeoff_results, RUN_MODE, RESUME, SHOW_PLOTS,
    name="multiplicity_burden_differentiable")
display(multiplicity_results)


## 10. Decision-tree duality gap
Use the original tree predictions and mixed-integer weighted-loss oracle. Quick uses three depth-two trees and 10 rounds. Feature-subset diversity is enabled by the same diversity flag. Both the averaged-feature gap and uniform-random-response gap are saved and plotted. The average-feature gap has no convex convergence guarantee; averaging tree responses can worsen predictions.


In [ ]:
if RUN_TREES:
    tree_gap_results = runtime.run_section("tree_duality_gap", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
    display(runtime.compact_summary(tree_gap_results))
else:
    tree_gap_results = None
    print("Disabled by RUN_TREES=False.")


## 11. Tree and random-forest baselines
Keep the V6 decision-tree, probability-averaging forest, and hard-voting forest comparisons. The configured selection rule applies to the optimization models. Methods include ORPM-tree, Joint-minimax, ADV/ROAR surrogates, constrained RobX, and OCEAN-style recourse; hard-voting forests also include DirectSAA and RobustSAA adaptations. Comparison ORPM-tree retains its best-feasible-candidate output rule.


In [ ]:
if RUN_TREES:
    tree_comparison_results = runtime.run_section("tree_comparison", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
    display(runtime.compact_summary(tree_comparison_results))
else:
    tree_comparison_results = None
    print("Disabled by RUN_TREES=False.")


## 12. Tree budget–validity trade-offs
Reuse each trained tree/forest set and the reference- or majority-selected cohort across its budget grid. Save per-dataset figures and tables; keep the existing baseline and best-candidate return rules.


In [ ]:
if RUN_TREES:
    tree_tradeoff_results = runtime.run_section("tree_tradeoff", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS)
    display(runtime.compact_summary(tree_tradeoff_results))
else:
    tree_tradeoff_results = None
    print("Disabled by RUN_TREES=False.")


## 13. Burden of multiplicity — trees
Use the saved tree trade-offs from the preceding cell. Singleton searches evaluate original predictions and retain solver bounds and statuses. Valid saved joint recommendations can serve as singleton witnesses; unresolved searches remain distinct from infeasibility certificates.


In [ ]:
if RUN_TREES:
    if tree_tradeoff_results is None:
        raise RuntimeError("Run the tree trade-off cell first.")
    tree_multiplicity_results = runtime.run_burden(tree_tradeoff_results, RUN_MODE, RESUME, SHOW_PLOTS,
        name="multiplicity_burden_tree")
    display(tree_multiplicity_results)
else:
    tree_multiplicity_results = None


## 14. Optional diversity-off/on comparison
Disabled initially to keep quick execution small. This runs the convex gap and mixed trade-offs in both diversity conditions. In reference mode, the selected individuals are the same. In majority mode, eligibility can change between conditions: paired differences use only individuals actually evaluated in both conditions, and the report includes overlap counts. If there is no overlap, the paired estimate is unavailable. Across-condition gap comparisons use raw cross-entropy units, avoiding different normalization scales.


In [ ]:
if RUN_PAIRED_DIVERSITY_CHECK:
    from utils import recourse_diversity_study as diversity_study
    diversity_comparison = diversity_study.run_paired(mode=RUN_MODE, seed=SEED, resume=RESUME, show=SHOW_PLOTS)
    display(diversity_comparison["summary"])
    print("Comparison directory:", diversity_comparison["directory"])
else:
    print("Optional extra diversity comparison disabled.")


## 15. Saved outputs and interpretation
The package includes a `results/` folder beside `dataset/`, `utils/`, and this notebook. All new results are saved there. Outputs use `results/V7/<mode>/<reference-or-majority>/<diversity-condition>/<experiment>/<run-id>/`. Figures and tables are grouped by dataset in `figures/<dataset>/` and `table/<dataset>/`. Burden outputs use the source selection rule and their own run IDs.

Quick mode checks operation; two seeds and three individuals are exploratory. Statistical intervals use equally weighted seed means, not pooled instances. Zero observed seed variance is flagged. Gap bounds measure optimization uncertainty and are distinct from those intervals. Feature feasibility uses the inherited continuous actionability relaxation. Target encodings and their audit notes are unchanged from V6; see `dataset/README.txt`.

To change the cohort rule or run mode, edit the control cell and rerun it and the experiment cells (or restart and run all). Saved outputs remain separate. V4/V5 results are optional read-only historical inputs; this notebook does not overwrite or automatically consume them.


In [ ]:
display(runtime.run_index())
print("Results root:", ROOT / "results" / "V7" / campaign.mode_name(RUN_MODE) / CANDIDATE_SELECTION)
